# AliasingAtlas: Time & Frequency Domain Sampling Interactive Tool

This interactive notebook demonstrates the principles of signal sampling, the **Nyquist-Shannon Sampling Theorem**, aliasing artifacts, and modern discrete signal reconstruction (Zero-Order Hold vs. Perfect FFT/Sinc Interpolation).

### Instructions for Google Colab:
1. Click the **Play** button on the code cell below to initialize the interactive workspace.
2. Use the interactive browser sliders to dynamically modify the target **Signal Frequency** and the **Sampling Rate**.
3. Keep an eye out for the automatic **Aliasing Warning** alert when your sampling frequency drops below the Nyquist limit ($f_{sampling} < 2 \cdot f_{signal}$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from typing import Tuple, Union

# Ensure plots render natively inline inside the Google Colab viewport
%matplotlib inline

def calculate_data(f_signal: float, f_sampling: float, bits: int, window_type: str, phase_val: float, wave_type: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, float, float]:
    """Generates continuous, sampled, and reconstructed signal data.

    Args:
        f_signal (float): Frequency of the source sine wave in Hz.
        f_sampling (float): Frequency at which to sample the signal in Hz.
        bits (int): Bit depth for amplitude quantization.
        window_type (str): Type of window function ('None', 'Hamming', 'Hann').
        phase_val (float): Phase offset in radians.
        wave_type (str): Type of waveform ('Sine', 'Square', 'Sawtooth').

    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, float, float]:
            (t_cont, y_cont, t_samp, y_samp, y_recon, freqs, mags, rmse_fft)
            t_cont (np.ndarray): Time axis for continuous signal.
            y_cont (np.ndarray): Values of the continuous signal.
            t_samp (np.ndarray): Time axis for sampled signal.
            y_samp (np.ndarray): Values of the sampled signal.
            y_recon (np.ndarray): Reconstructed signal values.
            freqs (np.ndarray): Frequency bins for spectrum.
            mags (np.ndarray): Magnitude spectrum values.
            rmse_fft (float): RMSE between original and reconstructed.

    Raises:
        TypeError: If inputs are not numeric.
        ValueError: If frequencies are not positive.
    """
    # Input Validation
    if not isinstance(f_signal, (int, float, np.number)) or not isinstance(f_sampling, (int, float, np.number)):
        raise TypeError("Frequencies must be numeric values.")
    if f_signal <= 0:
        raise ValueError("Signal frequency must be greater than zero.")
    if f_sampling <= 0:
        raise ValueError("Sampling frequency must be greater than zero.")

    # 1. Setup Parameters
    duration = 3 / f_signal  # Show exactly 3 full cycles
    num_continuous_points = 1000
    
    # 2. Continuous Reference Signal
    t_cont = np.linspace(0, duration, num_continuous_points)
    if wave_type == 'Sine':
        y_cont = np.sin(2 * np.pi * f_signal * t_cont + phase_val)
    elif wave_type == 'Square':
        y_cont = np.sign(np.sin(2 * np.pi * f_signal * t_cont + phase_val))
    else: # Sawtooth
        y_cont = 2 * (f_signal * t_cont + phase_val/(2*np.pi) - np.floor(0.5 + f_signal * t_cont + phase_val/(2*np.pi)))
    
    # 3. Discretely Sampled Signal
    num_samples = int(np.floor(duration * f_sampling))
    if num_samples < 2: num_samples = 2 # Prevent edge-case mathematical division errors
    
    t_samp = np.linspace(0, (num_samples - 1) / f_sampling, num_samples)
    if wave_type == 'Sine':
        y_samp = np.sin(2 * np.pi * f_signal * t_samp + phase_val)
    elif wave_type == 'Square':
        y_samp = np.sign(np.sin(2 * np.pi * f_signal * t_samp + phase_val))
    else: # Sawtooth
        y_samp = 2 * (f_signal * t_samp + phase_val/(2*np.pi) - np.floor(0.5 + f_signal * t_samp + phase_val/(2*np.pi)))
    
    # 4. Bit-Depth Quantization
    levels = 2**bits
    y_samp = np.round((y_samp + 1) / 2 * (levels - 1)) / (levels - 1) * 2 - 1

    # 5. Windowing Application
    N_samp = len(y_samp)
    if window_type == 'Hamming':
        window = np.hamming(N_samp)
    elif window_type == 'Hann':
        window = np.hanning(N_samp)
    else:
        window = np.ones(N_samp)
    
    y_samp_fft_input = y_samp * window

    # 6. Discrete Fourier Transform Reconstruction (Sinc Interpolation)
    Y_fft = np.fft.fft(y_samp_fft_input)
    
    Y_padded = np.zeros(num_continuous_points, dtype=complex)
    half = (N_samp + 1) // 2
    Y_padded[:half] = Y_fft[:half]
    Y_padded[num_continuous_points - (N_samp // 2):] = Y_fft[N_samp - (N_samp // 2):]
    
    y_recon = np.fft.ifft(Y_padded).real * (num_continuous_points / N_samp)
    
    # 7. Frequency Domain Spectrum Mapping
    freqs = np.fft.rfftfreq(N_samp, 1/f_sampling)
    mags = np.abs(np.fft.rfft(y_samp_fft_input)) / N_samp
    
    # 8. Mathematical Reconstruction Error Calculation (RMSE)
    noise = y_cont - y_recon
    rmse_fft = np.sqrt(np.mean(noise**2))
    signal_p = np.mean(y_cont**2)
    noise_p = np.mean(noise**2)
    snr = 10 * np.log10(signal_p / noise_p) if noise_p > 0 else 100
    
    return t_cont, y_cont, t_samp, y_samp, y_recon, freqs, mags, rmse_fft, snr

def update_plot(f_sig: float, f_samp: float, bits: int, window: str, phase: float, wave: str) -> None:
    """Updates the plot visualization based on user-provided frequencies.

    Args:
        f_sig (float): Signal frequency in Hz.
        f_samp (float): Sampling frequency in Hz.
        bits (int): Quantization bit depth.
        window (str): Window function type.
        phase (float): Phase offset.
        wave (str): Waveform type.

    Raises:
        TypeError: If inputs are not numeric.
        ValueError: If frequencies are not positive.
    """
    # Input Validation
    if not isinstance(f_sig, (int, float, np.number)) or not isinstance(f_samp, (int, float, np.number)):
        raise TypeError("Frequencies must be numeric values.")
    if f_sig <= 0 or f_samp <= 0:
        # In interactive context, return early to avoid breaking widget state
        return

    # Core mathematical rendering framework optimized for browser canvas redraws
    t_cont, y_cont, t_samp, y_samp, y_recon, freqs, mags, err, snr = calculate_data(f_sig, f_samp, bits, window, phase, wave)
    
    fig, (ax_time, ax_freq) = plt.subplots(2, 1, figsize=(13, 8))
    plt.subplots_adjust(hspace=0.35)
    
    # --- Time Domain Plotting Layer ---
    ax_time.plot(t_cont, y_cont, 'b--', alpha=0.4, label='Ideal Continuous')
    ax_time.step(t_samp, y_samp, where='post', color='crimson', alpha=0.6, label='Zero-Order Hold')
    ax_time.plot(t_cont, y_recon, 'g-', linewidth=2, label='FFT Reconstruction')
    ax_time.scatter(t_samp, y_samp, color='darkred', s=25, zorder=3, label='Sample Points')
    
    ax_time.set_title("AliasingAtlas: Time Domain Sampling & Reconstruction", fontweight='bold', fontsize=12)
    ax_time.set_xlabel("Time (seconds)")
    ax_time.set_ylabel("Amplitude")
    ax_time.set_xlim(0, 3/f_sig)
    ax_time.set_ylim(-1.3, 1.3)
    ax_time.legend(loc='upper right', fontsize='small')
    ax_time.grid(True, linestyle=':', alpha=0.6)
    
    # --- Frequency Domain Plotting Layer ---
    ax_freq.plot(freqs, mags, 'ro-', markersize=4, linewidth=1.5, label='Signal Spectrum')
    ax_freq.axvline(f_samp/2, color='orange', linestyle='--', linewidth=2, label='Nyquist Limit')
    
    ax_freq.set_title("Frequency Domain: Magnitude Spectrum", fontweight='bold', fontsize=12)
    ax_freq.set_xlabel("Frequency (Hz)")
    ax_freq.set_ylabel("Magnitude")
    ax_freq.axvline(f_sig, color='blue', alpha=0.3, label='Target Frequency')
    ax_freq.set_xlim(0, max(f_samp * 0.6, f_sig * 1.5))
    ax_freq.set_ylim(0, max(mags.max() * 1.2, 0.1))
    ax_freq.legend(loc='upper right')
    ax_freq.grid(True, linestyle=':', alpha=0.6)
    
    # --- Status, Warnings & Dynamic HTML Banners ---
    is_aliased = f_samp < 2 * f_sig
    bg_color = "#ffebe6" if is_aliased else "#e6f9ed"
    border_color = "#ff4d4d" if is_aliased else "#2eb85c"
    text_color = "#cc0000" if is_aliased else "#1b5e20"
    warn_text = "⚠️ CRITICAL WARNING: ALIASING DETECTED!" if is_aliased else "✅ Sampling Conditions Valid (No Aliasing)"
    
    status_html = f"""
    <div style="background-color: {bg_color}; border: 2px solid {border_color}; padding: 10px; border-radius: 5px; text-align: center; font-family: sans-serif;">
        <strong style="color: {text_color}; font-size: 14px;">{warn_text}</strong><br>
        <span style="color: #333; font-size: 13px;">RMSE: <b>{err:.4f}</b> | SNR: <b>{snr:.1f} dB</b></span>
    </div>
    """
    display(widgets.HTML(status_html))
    plt.show()

# Deploy native browser-rendered interactive controls
interact(
    update_plot,
    f_sig = widgets.FloatSlider(description='Signal (Hz):', min=1.0, max=50.0, step=0.5, value=10.0, continuous_update=False),
    f_samp = widgets.FloatSlider(description='Sampling (Hz):', min=5.0, max=1500.0, step=5.0, value=50.0, continuous_update=False),
    bits = widgets.IntSlider(description='Bit Depth:', min=2, max=16, step=1, value=16, continuous_update=False),
    window = widgets.Dropdown(options=['None', 'Hamming', 'Hann'], value='None', description='Window:'),
    phase = widgets.FloatSlider(description='Phase:', min=0.0, max=2*np.pi, step=0.1, value=np.pi/4, continuous_update=False),
    wave = widgets.Dropdown(options=['Sine', 'Square', 'Sawtooth'], value='Sine', description='Waveform:')
);